# item15 오류대응 텍스트 골드셋

실습 골드셋의 각 실습 스팬 안에서 **오류대응 텍스트**를 LLM으로 추출한다.

흐름:
1. 날짜별 실습 골드셋 final 로드
2. 각 실습 스팬의 문장을 LLM에 전달
3. LLM이 오류대응 서브스팬(`start_index`~`end_index`) 추출
4. 날짜별 검수용 `*_error_goldset_v2.json/xlsx` 저장
5. 사람 검수 후 확정 → `*_error_goldset_final.json`

In [7]:
import json
import re
from pathlib import Path

import pandas as pd
import google.generativeai as genai
from tqdm.auto import tqdm

import sys

# app/ 디렉토리가 있는 프로젝트 루트를 찾아 sys.path에 추가
_p = Path(".").resolve()
while not (_p / "app").exists() and _p != _p.parent:
    _p = _p.parent
sys.path.insert(0, str(_p))

from app.core.config import settings

genai.configure(api_key=settings.api_key)
MODEL = settings.eval_model
DATES = [f"2026-02-{day:02d}" for day in range(9, 14)]
GS       = _p / "data/goldset"
GS_LABEL = _p / "data/goldset/labeling"
CSV_PATH = _p / "data/processed/lectures_kss.csv"

df_all = pd.read_csv(CSV_PATH)
print(f"프로젝트 루트: {_p}")
print(f"대상 날짜: {', '.join(DATES)}")
print(f"model: {MODEL}")
for date in DATES:
    n = int((df_all["date"] == date).sum())
    print(f"  {date}: 문장 {n}개")

ModuleNotFoundError: No module named 'app.core'

In [25]:
# ── 날짜별 실습 골드셋 로드 ──────────────────────────────────────────
def load_practices(date):
    path = GS_LABEL / f"{date}_practice_goldset_final.json"
    data = json.loads(path.read_text(encoding="utf-8"))
    practices = data["practices"]
    print(f"[{date}]  실습 {len(practices)}개")
    return practices

---

## 프롬프트 설정

**오류대응** 정의를 아래 프롬프트에서 직접 수정하세요.

In [26]:
ERR_SYSTEM = """당신은 강의 전사 텍스트에서 '오류대응' 구간을 식별하는 전문가입니다.
응답은 반드시 JSON만 출력하세요."""

ERR_USER = """아래는 한 실습 구간의 강의 전사입니다. 각 줄은 [전역인덱스] 문장 형식입니다.

이 구간에서 **오류대응** 텍스트를 식별하세요.

【오류대응 정의】
실습 진행 중 실제로 오류/문제가 발생한 상황과 그 직접적인 대응을 포함하는 구간입니다.
다음 조건을 모두 만족해야 합니다:
① 오류/문제가 이 구간 내에서 실제로 발생했을 것 (과거 언급·가능성 언급 아님)
② 그 오류는 SQL·DB·쿼리 실행과 직접 관련된 것일 것
③ 아래 세 유형 중 하나에 해당할 것:
   - 수강생이 실습 중 발생한 오류(에러 메시지, 예상과 다른 결과 등)에 대해 질문하는 상황
   - 강사가 실습 진행 중 직접 오류를 경험하는 상황 (쿼리 실패, 예상과 다른 결과 발생)
   - 바로 앞에서 실제 오류가 발생한 직후 강사가 원인을 설명하거나 해결하는 상황

【반드시 제외】
- 오류 없이 정상 진행 중 개념 설명으로 오류 케이스를 예시로 언급하는 경우
- SQL/DB와 무관한 도구 문제 (마우스, 키보드, IDE 화면 표시, 자동 정렬 등)
- 데이터 탐색·무결성 확인 과정에서 NULL·빈값을 조회하는 정상 실습
- 오류 발생 가능성만 언급하는 경우 (실제 발생 아님)
- 오류 없이 성공적으로 완료된 후 결과를 설명하는 구간
- 단순 진행 안내, 잡담

각 오류대응 구간에 대해:
- error_name: 어떤 오류/문제 상황인지 짧은 이름
- start_index / end_index: 시작/끝 문장의 전역 인덱스 (반드시 아래 구간 내 값)
- key_sentence: 오류 발생 또는 대응을 가장 잘 나타내는 핵심 문장 원문 1개

오류대응이 없으면 {{"errors": []}} 로 응답하세요.

문장 구간:
{indexed_sentences}

아래 JSON 형식으로만 응답하세요:
{{"errors": [{{"error_name": "...", "start_index": 0, "end_index": 0, "key_sentence": "..."}}]}}"""


def _format_span(sentences, start, end):
    return "\n".join(f"[{i}] {sentences[i]}" for i in range(start, end + 1))


def _parse_errors(raw):
    text = re.sub(r"^```\w*\s*", "", raw.strip())
    text = re.sub(r"\s*```$", "", text).strip()
    try:
        data = json.loads(text)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", text, re.DOTALL)
        if not m:
            return []
        data = json.loads(m.group())
    return data.get("errors", [])


def run_error_extraction(practices, sentences, df_date, date):
    model = genai.GenerativeModel(MODEL)
    results = []
    for p in tqdm(practices, desc=f"error-extract {date}"):
        s, e = p["start"], p["end"]
        prompt = f"{ERR_SYSTEM}\n\n{ERR_USER.format(indexed_sentences=_format_span(sentences, s, e))}"
        resp = model.generate_content(
            prompt,
            generation_config=genai.GenerationConfig(
                response_mime_type="application/json",
                temperature=settings.llm_temperature,
            )
        )
        for err in _parse_errors(resp.text):
            try:
                es, ee = int(err["start_index"]), int(err["end_index"])
            except (KeyError, ValueError, TypeError):
                continue
            if es > ee:
                es, ee = ee, es
            es, ee = max(es, s), min(ee, e)
            if es > ee:
                continue
            results.append({
                "error_name":     str(err.get("error_name", "")).strip(),
                "start":          es,
                "end":            ee,
                "key_sentence":   str(err.get("key_sentence", "")).strip(),
                "from_practice":  p["practice_name"],
                "from_topic":     p.get("from_topic", ""),
                "timestamp":      df_date.loc[es, "timestamp"],
                "text":           " ".join(sentences[i] for i in range(es, ee + 1)),
            })
    return results

In [27]:
# ── 5개 날짜 v2 초안 저장 (사람 검수용) ────────────────────────────────
import openpyxl
from openpyxl.styles import Alignment


def save_draft(date, errors_raw):
    draft = {
        "source": CSV_PATH.name,
        "date": date,
        "method": "llm_extraction_from_practice_spans",
        "error_count": len(errors_raw),
        "errors": errors_raw,
    }
    draft_json = GS / f"{date}_error_goldset_v2.json"
    draft_json.write_text(json.dumps(draft, ensure_ascii=False, indent=2), encoding="utf-8")

    rows = [{"timestamp": e["timestamp"], "error_name": e["error_name"],
             "span": f"{e['start']}~{e['end']}",
             "from_practice": e.get("from_practice", ""),
             "from_topic": e.get("from_topic", ""),
             "key_sentence": e["key_sentence"], "text": e["text"], "human_check": ""}
            for e in errors_raw]
    xlsx_path = GS / f"{date}_error_goldset_v2.xlsx"
    pd.DataFrame(rows).to_excel(xlsx_path, index=False)
    wb = openpyxl.load_workbook(xlsx_path); ws = wb.active
    for col, w in {"A": 12, "B": 35, "C": 12, "D": 50, "E": 30, "F": 50, "G": 70, "H": 12}.items():
        ws.column_dimensions[col].width = w
    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.alignment = Alignment(wrap_text=True, vertical="top")
    wb.save(xlsx_path)
    print(f"  저장: {draft_json.name}  ({len(errors_raw)}개)")
    return draft_json, xlsx_path


all_error_results = {}
for date in DATES:
    df_date = df_all[df_all["date"] == date].reset_index(drop=True)
    if df_date.empty:
        print(f"\n[{date}] 데이터 없음 → 건너뜀")
        continue

    sentences = df_date["text_raw"].fillna("").tolist()
    practices = load_practices(date)
    errors_raw = run_error_extraction(practices, sentences, df_date, date)
    all_error_results[date] = errors_raw

    print(f"  오류대응 {len(errors_raw)}개 추출")
    for err in errors_raw:
        print(f"  [{err['start']:4d}~{err['end']:4d}] {err['error_name'][:50]}  ← {err['from_practice'][:35]}")
    save_draft(date, errors_raw)
    print()

print("검수 표기: O=오류대응맞음 / X=아님 / M=나눠야")

[2026-02-09]  실습 22개


error-extract 2026-02-09: 100%|██████████| 22/22 [04:40<00:00, 12.75s/it]


  오류대응 8개 추출
  [ 104~ 109] CONCAT_WS 실행 시 내용 누락  ← 이름과 직업 연결 과제
  [ 192~ 192] 쿼리 실행 오타  ← Q3 문자열 조작 과제
  [ 316~ 316] 쿼리 작성 중 오타 발생  ← 조건부 RPAD 적용 과제 (IF, LENGTH 함수 활용)
  [ 468~ 478] INSERT 중복 키 오류  ← Q24 REPLACE 연습용 테이블 생성
  [ 811~ 813] 파일 경로 구분자 오류  ← Q27 LOAD_FILE 함수로 파일 로드하기
  [ 837~ 838] BLOB 데이터 NULL 입력  ← 파일 로드 결과(BLOB) 확인 과제
  [ 967~ 974] LONGTEXT 컬럼에 이미지 파일 삽입 오류  ← 파일 로드 및 LONGTEXT 테이블 삽입 과제
  [1196~1199] 미설치 Collation 사용 시 오류 발생  ← Q33 중국어 데이터 처리 및 정렬 과제
  저장: 2026-02-09_error_goldset_v2.json  (8개)

[2026-02-10]  실습 26개


error-extract 2026-02-10: 100%|██████████| 26/26 [06:12<00:00, 14.32s/it]


  오류대응 9개 추출
  [ 614~ 617] 쿼리 결과 NULL 값 확인 및 원인 설명  ← RIGHT OUTER JOIN으로 모든 부서와 사원 조회
  [ 704~ 705] 셀프 조인 결과 누락  ← 6번. Self Join으로 사원과 매니저 출력
  [ 736~ 741] 조인 결과 행 개수 불일치  ← ANSI 조인을 이용한 관리자 이름 출력 (NULL 처리)
  [1233~1237] 버전별 실행 계획 결과 차이  ← EXPLAIN으로 JOIN 쿼리 실행 계획 분석
  [1289~1290] 복사/붙여넣기 형식 오류  ← 인덱스 유무에 따른 쿼리 비용 비교 확인
  [1573~1580] 쿼리 작성 오류 및 수정  ← City 테이블 상위 10개 도시 조회 및 전체 개수 확인
  [1661~1664] 문자열 컬럼 COUNT 감소 원인 분석  ← Q2: Country 테이블 카운트 집계로 무결성 확인
  [1735~1736] 오라클 구문 사용 오류  ← 모든 국가와 수도 이름 출력 (LEFT OUTER JOIN)
  [1754~1755] 쿼리 수정 후 정상 작동  ← 모든 국가와 수도 이름 출력 (LEFT OUTER JOIN)
  저장: 2026-02-10_error_goldset_v2.json  (9개)

[2026-02-11]  실습 27개


...

  오류대응 12개 추출
  [ 387~ 419] UNION 컬럼 개수/타입 불일치 오류  ← Q5-1: 컬럼 수가 다른 테이블 병합 시 오류 해결
  [ 601~ 602] 에디터 구문 오류 표시  ← 차집합을 이용해 사원이 없는 부서 번호 출력
  [ 751~ 767] 다중행 서브쿼리 오류  ← Q2. 세일즈맨과 같은 월급을 받는 사원 조회
  [ 955~ 956] NOT IN 연산 시 NULL 값으로 인한 예상과 다른 결과  ← 부하직원이 없는 사원의 사원번호와 이름 출력
  [1069~1075] 쿼리 논리 오류 수정 및 설명  ← 10번 과제: 20번 부서 최저 월급자보다 더 많이 받는 사원 
  [1134~1135] 쿼리 결과 없음  ← 멀티 컬럼 서브쿼리 과제: 세일즈맨과 부서, 월급이 같은 사원 
  [1194~1195] 잘못된 쿼리로 결과 없음  ← Q14. 다중열 서브쿼리: 부서별 최고 월급 사원 조회
  [1261~1263] 쿼리 작성 중 함수 오기입 수정  ← 상관 서브쿼리를 이용한 부서 평균 급여와의 차이 계산
  [1405~1405] 쿼리 작성 중 구문 오류 인지  ← 사원 이름과 모든 사원 봉급의 합 출력
  [1433~1451] WHERE절에서 SELECT 별칭 사용 오류  ← SELECT에서 계산된 결과(별칭)를 WHERE절에서 사용하기
  [1566~1567] 쿼리 결과 중복 발생  ← JOIN을 사용하여 월급여 3천 이상 사원이 있는 부서 정보 출
  [1617~1618] 데이터 값 대소문자 구분 오류  ← 인구수 100만 이상 도시의 이름, 인구, 국가 이름 출력 (과
  저장: 2026-02-11_error_goldset_v2.json  (12개)

[2026-02-12]  실습 14개


error-extract 2026-02-12: 100%|██████████| 14/14 [03:19<00:00, 14.25s/it]


  오류대응 3개 추출
  [ 244~ 247] 쿼리 실행 결과 없음  ← TABLE 키워드와 서브쿼리 실행
  [ 331~ 337] 쿼리 실행 결과 없음  ← ANY, SOME, IN 연산자 예제 실행
  [ 719~ 719] 예상과 다른 쿼리 결과  ← Q17: 서브쿼리를 CTE로 재구성하기
  저장: 2026-02-12_error_goldset_v2.json  (3개)

[2026-02-13]  실습 24개


error-extract 2026-02-13: 100%|██████████| 24/24 [05:23<00:00, 13.46s/it]

  오류대응 11개 추출
  [ 103~ 119] 테이블 삭제(DROP) 실패  ← Q8 테이블 삭제 실습
  [ 591~ 592] 롤백이 동작하지 않는 문제  ← 체크 제약조건 추가 및 UPDATE/ROLLBACK 실습
  [ 678~ 679] 체크 제약 조건 위반  ← 체크 제약 조건 수정 및 INSERT 테스트
  [ 680~ 681] 키 제약 조건 위반  ← 체크 제약 조건 수정 및 INSERT 테스트
  [ 952~ 968] 새 세션의 오토커밋 설정값 불일치  ← 시스템 쿼리를 이용한 오토커밋 설정 변경
  [1043~1054] 동일 계정 접속으로 인한 트랜잭션 미반영  ← 두 세션 간 트랜잭션 결과 확인
  [1089~1090] 데이터 충돌 오류  ← 세션 간 데이터베이스 락(Lock) 현상 테스트
  [1207~1207] 실행 중 오류 발생 인지 및 해결 안내  ← 양쪽 세션에서 트랜잭션 및 롤백 독립성 확인
  [1302~1307] DBMS 연결 끊김  ← Q1: 사원 월급 변경
  [1324~1330] UPDATE 쿼리 실행 오류  ← 부서번호 20번 사원 월급 2000으로 변경
  [1407~1407] 명령어 실행 오류  ← 서브쿼리를 이용한 부서번호 변경
  저장: 2026-02-13_error_goldset_v2.json  (11개)

검수 표기: O=오류대응맞음 / X=아님 / M=나눠야


---

## 확정 — 검수 결과 반영

각 날짜의 `*_error_goldset_v2.xlsx`에서 `human_check` 열을 채운 뒤 아래 셀을 실행해 확정본을 저장한다.
- `DROP`: 제외할 항목의 start 인덱스와 사유
- `DROP_BY_DATE`: 날짜별로 다른 DROP 규칙을 줄 때 사용


In [8]:
import json
import pandas as pd
from pathlib import Path

# 프로젝트 루트 자동 탐색 (data/goldset 기준)
_p = Path(".").resolve()
while not (_p / "data" / "goldset").exists() and _p != _p.parent:
    _p = _p.parent

DATES = [f"2026-02-{day:02d}" for day in range(9, 14)]
GS = _p / "data" / "goldset"

# ── 5개 날짜 확정: O 항목만 남기고 DROP 제외 ────────────────────────────
DROP_BY_DATE = {
    # "2026-02-09": {123: "사유"},
}

for date in DATES:
    xlsx_path = GS / f"{date}_error_goldset_v2.xlsx"
    json_path = GS / f"{date}_error_goldset_v2.json"
    if not xlsx_path.exists() or not json_path.exists():
        print(f"[{date}] v2 파일 없음 → 건너뜀")
        continue

    review = pd.read_excel(xlsx_path)
    v2 = json.loads(json_path.read_text(encoding="utf-8"))
    by_span = {(e["start"], e["end"]): e for e in v2["errors"]}
    drop = DROP_BY_DATE.get(date, {})

    final, excluded = [], []
    for _, r in review.iterrows():
        hc = str(r["human_check"]).strip().upper()
        if hc != "O":
            continue
        s, e = map(int, str(r["span"]).split("~"))
        item = by_span.get((s, e))
        if not item:
            continue
        if s in drop:
            item["_reason"] = drop[s]
            excluded.append(item)
        else:
            final.append(item)
    final.sort(key=lambda x: x["start"])

    final_obj = {
        "source": v2["source"], "date": date,
        "method": "human_reviewed", "error_count": len(final),
        "errors": final,
        "excluded": [{"start": d["start"], "error_name": d["error_name"], "reason": d.get("_reason", "")} for d in excluded],
    }
    final_json = GS / f"{date}_error_goldset_final.json"
    final_json.write_text(json.dumps(final_obj, ensure_ascii=False, indent=2), encoding="utf-8")

    print(f"[{date}] 확정 {len(final)}개 / 제외 {len(excluded)}개 → {final_json.name}")
    for err in final:
        print(f"  [{err['start']:4d}~{err['end']:4d}] {err['error_name'][:50]}")

[2026-02-09] 확정 5개 / 제외 0개 → 2026-02-09_error_goldset_final.json
  [ 104~ 109] CONCAT_WS 실행 시 내용 누락
  [ 468~ 478] INSERT 중복 키 오류
  [ 837~ 838] BLOB 데이터 NULL 입력
  [ 967~ 974] LONGTEXT 컬럼에 이미지 파일 삽입 오류
  [1196~1199] 미설치 Collation 사용 시 오류 발생
[2026-02-10] 확정 3개 / 제외 0개 → 2026-02-10_error_goldset_final.json
  [ 704~ 705] 셀프 조인 결과 누락
  [ 736~ 741] 조인 결과 행 개수 불일치
  [1735~1736] 오라클 구문 사용 오류
[2026-02-11] 확정 10개 / 제외 0개 → 2026-02-11_error_goldset_final.json
  [ 387~ 419] UNION 컬럼 개수/타입 불일치 오류
  [ 751~ 767] 다중행 서브쿼리 오류
  [ 955~ 956] NOT IN 연산 시 NULL 값으로 인한 예상과 다른 결과
  [1069~1075] 쿼리 논리 오류 수정 및 설명
  [1194~1195] 잘못된 쿼리로 결과 없음
  [1261~1263] 쿼리 작성 중 함수 오기입 수정
  [1405~1405] 쿼리 작성 중 구문 오류 인지
  [1433~1451] WHERE절에서 SELECT 별칭 사용 오류
  [1566~1567] 쿼리 결과 중복 발생
  [1617~1618] 데이터 값 대소문자 구분 오류
[2026-02-12] 확정 2개 / 제외 0개 → 2026-02-12_error_goldset_final.json
  [ 244~ 247] 쿼리 실행 결과 없음
  [ 719~ 719] 예상과 다른 쿼리 결과
[2026-02-13] 확정 13개 / 제외 0개 → 2026-02-13_error_goldset_final.json
  [ 103~ 119] 테이블 삭제(DROP) 실패
  [ 59